In [2]:
import pandas as pd

df = pd.read_csv('../data/loans_clean.csv')
print("Filas:", len(df))
df.head()

Filas: 89785


,Loan ID,Customer ID,Loan Status,Current Loan Amount,Term,Credit Score,Annual Income,Years in current job,Home Ownership,Purpose,Monthly Debt,Years of Credit History,Number of Open Accounts,Number of Credit Problems,Current Credit Balance,Maximum Open Credit,Bankruptcies,Tax Liens
0,14dd8831-6af5-400b-83ec-68e61888a048,981165ec-3274-42f5-a3b4-d104041a9ca9,Fully Paid,445412,Short Term,709,1167493,8 years,Home Mortgage,Home Improvements,5215,17.2,6.0,1.0,228190,416746,1.0,0.0
1,4771cc26-131a-45db-b5aa-537ea4ba5342,2de017a3-2e01-49cb-a581-08169e83be29,Fully Paid,262328,Short Term,729,1168975,10+ years,Home Mortgage,Debt Consolidation,33296,21.1,35.0,0.0,229976,850784,0.0,0.0
2,4eed4e6a-aa2f-4c91-8651-ce984ee8fb26,5efb2b2b-bf11-4dfd-a572-3761a2694725,Fully Paid,266244,Short Term,741,2231892,8 years,Own Home,Debt Consolidation,29201,14.9,18.0,1.0,297996,750090,0.0,0.0
3,77598f7b-32e7-4e3b-a6e5-06ba0d98fe8a,e777faab-98ae-45af-9a86-7ce5b33b1011,Fully Paid,347666,Long Term,721,806949,3 years,Own Home,Debt Consolidation,8742,12.0,9.0,0.0,256329,386958,0.0,0.0
4,d4062e70-befa-4995-8643-a0de73938182,81536ad9-5ccf-4eb8-befb-47a4d608658e,Fully Paid,176220,Short Term,729,1168975,5 years,Rent,Debt Consolidation,20640,6.1,15.0,0.0,253460,427174,0.0,0.0


In [3]:
import sqlite3

conn = sqlite3.connect('../data/loans_model.db')
df.to_sql('loans', conn, if_exists='replace', index=False)

print("Base de datos creada con éxito.")

Base de datos creada con éxito.


In [4]:
query = """
WITH loan_features AS (
    SELECT
        "Loan ID",
        "Loan Status",
        Purpose,
        "Credit Score",
        ROUND(CAST("Current Loan Amount" AS FLOAT) / "Annual Income", 3) AS ratio_prestamo_ingreso,
        ROUND(CAST("Monthly Debt" AS FLOAT) / ("Annual Income" / 12.0), 3) AS ratio_deuda_ingreso_mensual,
        ROUND(AVG("Credit Score") OVER (PARTITION BY Purpose), 1) AS credit_score_promedio_purpose,
        ROUND("Credit Score" - AVG("Credit Score") OVER (PARTITION BY Purpose), 1) AS diferencia_vs_promedio_purpose
    FROM loans
)
SELECT * FROM loan_features
ORDER BY Purpose
LIMIT 10
"""

features_preview = pd.read_sql_query(query, conn)
print(features_preview)

                                Loan ID  Loan Status        Purpose  \
0  b762002d-4f8b-4eb0-a1c7-b1a9874fd997   Fully Paid  Business Loan   
1  6b68fc88-02db-442a-b5a1-c1d47e345c16   Fully Paid  Business Loan   
2  af030a3c-f806-40b4-b8bf-d28e2afcbe62  Charged Off  Business Loan   
3  95c49c97-c5d5-4800-931d-0f8fc88977ef  Charged Off  Business Loan   
4  6d53bc1f-1ade-40a2-a643-512a5fb17be4   Fully Paid  Business Loan   
5  c548ad40-8269-48b7-beba-c44a8956acf1   Fully Paid  Business Loan   
6  dfe474b6-7e66-4058-ae25-916a199e92c6  Charged Off  Business Loan   
7  345e009d-b098-43e8-811f-ba8964b0196b   Fully Paid  Business Loan   
8  92a8b16e-dfd3-4463-abd0-e3cd83aa05f8   Fully Paid  Business Loan   
9  519e816b-a18d-4d2c-83b2-ed95df0b77ec   Fully Paid  Business Loan   

   Credit Score  ratio_prestamo_ingreso  ratio_deuda_ingreso_mensual  \
0           737                   0.177                        0.248   
1           725                   0.096                        0.145   
2 

In [5]:
query_completa = """
WITH loan_features AS (
    SELECT
        "Loan ID",
        "Loan Status",
        Purpose,
        Term,
        "Home Ownership",
        "Years in current job",
        "Credit Score",
        "Annual Income",
        "Years of Credit History",
        "Number of Open Accounts",
        "Number of Credit Problems",
        "Bankruptcies",
        ROUND(CAST("Current Loan Amount" AS FLOAT) / "Annual Income", 3) AS ratio_prestamo_ingreso,
        ROUND(CAST("Monthly Debt" AS FLOAT) / ("Annual Income" / 12.0), 3) AS ratio_deuda_ingreso_mensual,
        ROUND(AVG("Credit Score") OVER (PARTITION BY Purpose), 1) AS credit_score_promedio_purpose,
        ROUND("Credit Score" - AVG("Credit Score") OVER (PARTITION BY Purpose), 1) AS diferencia_vs_promedio_purpose
    FROM loans
)
SELECT * FROM loan_features
"""

df_modelo = pd.read_sql_query(query_completa, conn)
print("Filas:", len(df_modelo))
print("Columnas:", df_modelo.columns.tolist())
df_modelo.head()


Filas: 89785
Columnas: ['Loan ID', 'Loan Status', 'Purpose', 'Term', 'Home Ownership', 'Years in current job', 'Credit Score', 'Annual Income', 'Years of Credit History', 'Number of Open Accounts', 'Number of Credit Problems', 'Bankruptcies', 'ratio_prestamo_ingreso', 'ratio_deuda_ingreso_mensual', 'credit_score_promedio_purpose', 'diferencia_vs_promedio_purpose']


,Loan ID,Loan Status,Purpose,Term,Home Ownership,Years in current job,Credit Score,Annual Income,Years of Credit History,Number of Open Accounts,Number of Credit Problems,Bankruptcies,ratio_prestamo_ingreso,ratio_deuda_ingreso_mensual,credit_score_promedio_purpose,diferencia_vs_promedio_purpose
0,b762002d-4f8b-4eb0-a1c7-b1a9874fd997,Fully Paid,Business Loan,Short Term,Rent,6 years,737,1501912,18.0,8.0,0.0,0.0,0.177,0.248,708.9,28.1
1,6b68fc88-02db-442a-b5a1-c1d47e345c16,Fully Paid,Business Loan,Short Term,Home Mortgage,4 years,725,1386734,13.6,11.0,0.0,0.0,0.096,0.145,708.9,16.1
2,af030a3c-f806-40b4-b8bf-d28e2afcbe62,Charged Off,Business Loan,Short Term,Rent,2 years,741,666805,15.0,7.0,0.0,0.0,0.438,0.112,708.9,32.1
3,95c49c97-c5d5-4800-931d-0f8fc88977ef,Charged Off,Business Loan,Short Term,Own Home,1 year,681,573819,11.4,6.0,0.0,0.0,0.392,0.103,708.9,-27.9
4,6d53bc1f-1ade-40a2-a643-512a5fb17be4,Fully Paid,Business Loan,Short Term,Home Mortgage,4 years,695,679896,10.2,4.0,0.0,0.0,0.129,0.121,708.9,-13.9


In [6]:
df_modelo.to_csv('../data/loans_model_features.csv', index=False)
print("Guardado con éxito.")

Guardado con éxito.


In [7]:
# 1. Chequeo rápido de nulos antes de seguir (por las dudas con las divisiones que hicimos en SQL)
print("Nulos por columna:")
print(df_modelo.isnull().sum().sum())

Nulos por columna:
0


In [8]:
# Separamos el ID (no sirve para predecir) y el target (lo que queremos predecir)
X = df_modelo.drop(columns=['Loan ID', 'Loan Status'])
y = df_modelo['Loan Status']

print("Forma de X (variables predictoras):", X.shape)
print("Forma de y (variable objetivo):", y.shape)
print("\nValores únicos de y:", y.unique())


Forma de X (variables predictoras): (89785, 14)
Forma de y (variable objetivo): (89785,)

Valores únicos de y: <StringArray>
['Fully Paid', 'Charged Off']
Length: 2, dtype: str


In [9]:
y = y.map({'Fully Paid': 0, 'Charged Off': 1})
print(y.value_counts())

Loan Status
0    67146
1    22639
Name: count, dtype: int64


In [10]:
X = pd.get_dummies(X, drop_first=True)
print("Nueva forma de X:", X.shape)
X.head()

Nueva forma de X: (89785, 40)


,Credit Score,Annual Income,Years of Credit History,Number of Open Accounts,Number of Credit Problems,Bankruptcies,ratio_prestamo_ingreso,ratio_deuda_ingreso_mensual,credit_score_promedio_purpose,diferencia_vs_promedio_purpose,...,Years in current job_2 years,Years in current job_3 years,Years in current job_4 years,Years in current job_5 years,Years in current job_6 years,Years in current job_7 years,Years in current job_8 years,Years in current job_9 years,Years in current job_< 1 year,Years in current job_Unknown
0,737,1501912,18.0,8.0,0.0,0.0,0.177,0.248,708.9,28.1,...,False,False,False,False,True,False,False,False,False,False
1,725,1386734,13.6,11.0,0.0,0.0,0.096,0.145,708.9,16.1,...,False,False,True,False,False,False,False,False,False,False
2,741,666805,15.0,7.0,0.0,0.0,0.438,0.112,708.9,32.1,...,True,False,False,False,False,False,False,False,False,False
3,681,573819,11.4,6.0,0.0,0.0,0.392,0.103,708.9,-27.9,...,False,False,False,False,False,False,False,False,False,False
4,695,679896,10.2,4.0,0.0,0.0,0.129,0.121,708.9,-13.9,...,False,False,True,False,False,False,False,False,False,False


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("\nProporción de default en train:", y_train.mean().round(3))
print("Proporción de default en test:", y_test.mean().round(3))


Train: (71828, 40)
Test: (17957, 40)

Proporción de default en train: 0.252
Proporción de default en test: 0.252


In [12]:
from sklearn.linear_model import LogisticRegression

modelo = LogisticRegression(max_iter=1000, random_state=42)
modelo.fit(X_train, y_train)

print("Modelo entrenado con éxito.")

Modelo entrenado con éxito.


In [13]:
y_pred = modelo.predict(X_test)

print("Predicciones generadas:", len(y_pred))
print("Ejemplo de las primeras 10 predicciones:", y_pred[:10])
print("Valores reales correspondientes:    ", y_test.values[:10])

Predicciones generadas: 17957
Ejemplo de las primeras 10 predicciones: [0 0 0 0 0 0 0 0 0 0]
Valores reales correspondientes:     [0 0 1 1 1 1 1 0 1 0]


In [14]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

print("Defaults que el modelo predijo:", y_pred.sum())
print("Defaults reales en test:", y_test.sum())
print()
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print()
print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=['Fully Paid', 'Charged Off']))

Defaults que el modelo predijo: 220
Defaults reales en test: 4528

Accuracy: 0.748

Matriz de confusión:
[[13321   108]
 [ 4416   112]]

              precision    recall  f1-score   support

  Fully Paid       0.75      0.99      0.85     13429
 Charged Off       0.51      0.02      0.05      4528

    accuracy                           0.75     17957
   macro avg       0.63      0.51      0.45     17957
weighted avg       0.69      0.75      0.65     17957



In [15]:
from sklearn.metrics import roc_auc_score

probs = modelo.predict_proba(X_test)[:, 1]

print("ROC-AUC:", round(roc_auc_score(y_test, probs), 3))
print("Probabilidad mínima:", round(probs.min(), 3))
print("Probabilidad máxima:", round(probs.max(), 3))
print("Probabilidad promedio:", round(probs.mean(), 3))

ROC-AUC: 0.619
Probabilidad mínima: 0.0
Probabilidad máxima: 0.665
Probabilidad promedio: 0.253


In [16]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

modelo2 = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
modelo2.fit(X_train_scaled, y_train)

y_pred2 = modelo2.predict(X_test_scaled)
probs2 = modelo2.predict_proba(X_test_scaled)[:, 1]

print("ROC-AUC:", round(roc_auc_score(y_test, probs2), 3))
print("Probabilidad mínima:", round(probs2.min(), 3))
print("Probabilidad máxima:", round(probs2.max(), 3))
print()
print(confusion_matrix(y_test, y_pred2))
print()
print(classification_report(y_test, y_pred2, target_names=['Fully Paid', 'Charged Off']))

ROC-AUC: 0.63
Probabilidad mínima: 0.011
Probabilidad máxima: 0.941

[[8562 4867]
 [2033 2495]]

              precision    recall  f1-score   support

  Fully Paid       0.81      0.64      0.71     13429
 Charged Off       0.34      0.55      0.42      4528

    accuracy                           0.62     17957
   macro avg       0.57      0.59      0.57     17957
weighted avg       0.69      0.62      0.64     17957



In [17]:
print("Probabilidad promedio del modelo 2:", round(probs2.mean(), 3))
print("Tasa real de default en test:", round(y_test.mean(), 3))
print()

coefs = pd.DataFrame({
    'variable': X_train.columns,
    'coeficiente': modelo2.coef_[0].round(3)
})
coefs['abs'] = coefs['coeficiente'].abs()
coefs = coefs.sort_values('abs', ascending=False).drop(columns='abs')

print(coefs.head(12).to_string(index=False))


Probabilidad promedio del modelo 2: 0.487
Tasa real de default en test: 0.252

                      variable  coeficiente
               Term_Short Term       -0.229
   ratio_deuda_ingreso_mensual        0.130
 credit_score_promedio_purpose       -0.126
        ratio_prestamo_ingreso        0.111
                 Annual Income       -0.107
  Years in current job_Unknown        0.095
                  Credit Score       -0.091
           Home Ownership_Rent        0.081
diferencia_vs_promedio_purpose       -0.076
  Home Ownership_Home Mortgage       -0.065
                 Purpose_other       -0.064
       Years of Credit History       -0.059


In [18]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=50,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

probs_rf_test = rf.predict_proba(X_test)[:, 1]
probs_rf_train = rf.predict_proba(X_train)[:, 1]

print("ROC-AUC en test:", round(roc_auc_score(y_test, probs_rf_test), 3))
print("ROC-AUC en train:", round(roc_auc_score(y_train, probs_rf_train), 3))

importancias = pd.DataFrame({
    'variable': X_train.columns,
    'importancia': rf.feature_importances_.round(3)
}).sort_values('importancia', ascending=False)

print()
print(importancias.head(12).to_string(index=False))

ROC-AUC en test: 0.652
ROC-AUC en train: 0.685

                      variable  importancia
                  Credit Score        0.187
        ratio_prestamo_ingreso        0.170
               Term_Short Term        0.157
diferencia_vs_promedio_purpose        0.156
                 Annual Income        0.104
   ratio_deuda_ingreso_mensual        0.071
       Years of Credit History        0.034
  Home Ownership_Home Mortgage        0.021
           Home Ownership_Rent        0.019
 credit_score_promedio_purpose        0.019
       Number of Open Accounts        0.018
  Years in current job_Unknown        0.013


In [19]:
print(df_modelo['Home Ownership'].value_counts())

Home Ownership
Home Mortgage    43548
Rent             37855
Own Home          8199
HaveMortgage       183
Name: count, dtype: int64


In [20]:
mediana_score = df_modelo['Credit Score'].median()
mediana_ingreso = df_modelo['Annual Income'].median()

imputados = (df_modelo['Credit Score'] == mediana_score) & (df_modelo['Annual Income'] == mediana_ingreso)

print("Filas del grupo 'imputado':", imputados.sum())
print("Tasa de default en ese grupo:", round(y[imputados].mean(), 3))
print("Tasa de default en el resto:", round(y[~imputados].mean(), 3))

Filas del grupo 'imputado': 19154
Tasa de default en ese grupo: 0.283
Tasa de default en el resto: 0.244


In [21]:
cols_sql = ['diferencia_vs_promedio_purpose', 'credit_score_promedio_purpose']

rf_sin = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=50,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf_sin.fit(X_train.drop(columns=cols_sql), y_train)

probs_sin = rf_sin.predict_proba(X_test.drop(columns=cols_sql))[:, 1]
print("ROC-AUC en test sin las 2 variables de SQL:", round(roc_auc_score(y_test, probs_sin), 3))
print("ROC-AUC en test con ellas:", round(roc_auc_score(y_test, probs_rf_test), 3))

ROC-AUC en test sin las 2 variables de SQL: 0.651
ROC-AUC en test con ellas: 0.652


In [22]:
# 1. Corregir la etiqueta duplicada de Home Ownership
df_final = df_modelo.copy()
df_final['Home Ownership'] = df_final['Home Ownership'].replace('HaveMortgage', 'Home Mortgage')

# 2. Marcar explícitamente las filas con score e ingreso imputados
mediana_score = df_final['Credit Score'].median()
mediana_ingreso = df_final['Annual Income'].median()
df_final['datos_imputados'] = (
    (df_final['Credit Score'] == mediana_score) & (df_final['Annual Income'] == mediana_ingreso)
).astype(int)

# 3. Sacar las 2 variables de SQL que no aportaron, más el ID y el target
X2 = df_final.drop(columns=['Loan ID', 'Loan Status',
                            'diferencia_vs_promedio_purpose', 'credit_score_promedio_purpose'])
y2 = df_final['Loan Status'].map({'Fully Paid': 0, 'Charged Off': 1})

# 4. One-hot encoding y split, igual que antes
X2 = pd.get_dummies(X2, drop_first=True)
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

# 5. Mismo Random Forest, con las variables corregidas
rf2 = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=50,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf2.fit(X2_train, y2_train)

probs_rf2 = rf2.predict_proba(X2_test)[:, 1]
print("ROC-AUC en test (versión corregida):", round(roc_auc_score(y2_test, probs_rf2), 3))
print("Columnas de X2:", X2.shape[1])

imp2 = pd.DataFrame({
    'variable': X2_train.columns,
    'importancia': rf2.feature_importances_.round(3)
}).sort_values('importancia', ascending=False)

print()
print(imp2.head(10).to_string(index=False))

ROC-AUC en test (versión corregida): 0.653
Columnas de X2: 38

                    variable  importancia
                Credit Score        0.242
      ratio_prestamo_ingreso        0.196
             Term_Short Term        0.189
               Annual Income        0.118
 ratio_deuda_ingreso_mensual        0.089
     Years of Credit History        0.042
         Home Ownership_Rent        0.031
     Number of Open Accounts        0.020
             datos_imputados        0.016
Years in current job_Unknown        0.015


In [23]:
# 1. ¿El modelo rinde igual con datos reales que con datos imputados?
mask = (X2_test['datos_imputados'] == 1).values

print("ROC-AUC solo filas imputadas:", round(roc_auc_score(y2_test.values[mask], probs_rf2[mask]), 3))
print("ROC-AUC solo filas con datos reales:", round(roc_auc_score(y2_test.values[~mask], probs_rf2[~mask]), 3))

# 2. ¿Las probabilidades son honestas?
print()
print("Probabilidad promedio de rf2 (con class_weight):", round(probs_rf2.mean(), 3))
print("Tasa real de default en test:", round(y2_test.mean(), 3))

rf3 = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=50,
    random_state=42, n_jobs=-1
)
rf3.fit(X2_train, y2_train)
probs_rf3 = rf3.predict_proba(X2_test)[:, 1]

print()
print("ROC-AUC sin class_weight:", round(roc_auc_score(y2_test, probs_rf3), 3))
print("Probabilidad promedio sin class_weight:", round(probs_rf3.mean(), 3))
print("Probabilidad máxima sin class_weight:", round(probs_rf3.max(), 3))

ROC-AUC solo filas imputadas: 0.606
ROC-AUC solo filas con datos reales: 0.663

Probabilidad promedio de rf2 (con class_weight): 0.483
Tasa real de default en test: 0.252

ROC-AUC sin class_weight: 0.654
Probabilidad promedio sin class_weight: 0.253
Probabilidad máxima sin class_weight: 0.534


In [24]:
tabla = pd.DataFrame({
    'prob': probs_rf3,
    'real': y2_test.values
})
tabla['grupo'] = pd.qcut(tabla['prob'], q=10, labels=False, duplicates='drop') + 1

resumen = tabla.groupby('grupo').agg(
    prestamos=('real', 'count'),
    prob_promedio=('prob', 'mean'),
    default_real=('real', 'mean')
).round(3)

print(resumen)

       prestamos  prob_promedio  default_real
grupo                                        
1           1796          0.118         0.097
2           1796          0.169         0.143
3           1795          0.197         0.184
4           1796          0.218         0.205
5           1796          0.237         0.232
6           1795          0.255         0.246
7           1796          0.274         0.269
8           1795          0.299         0.325
9           1796          0.345         0.359
10          1796          0.416         0.461


In [25]:
def nivel(p):
    if p < 0.20:
        return 'Bajo'
    elif p < 0.30:
        return 'Medio'
    else:
        return 'Alto'

tabla['nivel'] = tabla['prob'].apply(nivel)

resumen_niveles = tabla.groupby('nivel').agg(
    prestamos=('real', 'count'),
    prob_promedio=('prob', 'mean'),
    default_real=('real', 'mean')
).round(3).loc[['Bajo', 'Medio', 'Alto']]

print(resumen_niveles)

       prestamos  prob_promedio  default_real
nivel                                        
Bajo        4686          0.155         0.133
Medio       8907          0.248         0.245
Alto        4364          0.367         0.395


In [26]:
import joblib
import json

# 1. Guardar el modelo entrenado
joblib.dump(rf3, '../api/modelo_riesgo.joblib')

# 2. Guardar lo que la API necesita para usarlo bien
metadata = {
    'columnas': X2_train.columns.tolist(),
    'mediana_credit_score': float(mediana_score),
    'mediana_annual_income': float(mediana_ingreso),
    'cortes': {'bajo_max': 0.20, 'medio_max': 0.30},
    'default_historico': {k: float(v) for k, v in resumen_niveles['default_real'].items()}
}

with open('../api/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

# 3. Verificar que el modelo guardado da las mismas predicciones
modelo_cargado = joblib.load('../api/modelo_riesgo.joblib')
diferencia = abs(modelo_cargado.predict_proba(X2_test)[:, 1] - probs_rf3).max()
print("Diferencia máxima entre el modelo guardado y el original:", diferencia)

Diferencia máxima entre el modelo guardado y el original: 2.7755575615628914e-16


In [27]:
metadata['categorias'] = {
    'Purpose': sorted(df_final['Purpose'].unique().tolist()),
    'Term': sorted(df_final['Term'].unique().tolist()),
    'Home Ownership': sorted(df_final['Home Ownership'].unique().tolist()),
    'Years in current job': sorted(df_final['Years in current job'].unique().tolist()),
}

with open('../api/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print({k: len(v) for k, v in metadata['categorias'].items()})

{'Purpose': 16, 'Term': 2, 'Home Ownership': 3, 'Years in current job': 12}


In [28]:
columnas_rango = [
    'Credit Score', 'Annual Income', 'Years of Credit History',
    'Number of Open Accounts', 'ratio_prestamo_ingreso', 'ratio_deuda_ingreso_mensual',
]

metadata['rangos'] = {
    col: {
        'p1': float(df_final[col].quantile(0.01)),
        'p99': float(df_final[col].quantile(0.99)),
    }
    for col in columnas_rango
}

with open('../api/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

for col, r in metadata['rangos'].items():
    print(f"{col}: {r['p1']:.3f} a {r['p99']:.3f}")

Credit Score: 627.000 a 750.000
Annual Income: 370639.840 a 4200730.520
Years of Credit History: 6.500 a 40.300
Number of Open Accounts: 3.000 a 27.000
ratio_prestamo_ingreso: 0.033 a 0.644
ratio_deuda_ingreso_mensual: 0.014 a 0.416
